In [49]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
import joblib

scaler = MinMaxScaler()
df_courses_tasks = pd.read_csv("test/courses_tasks_test.csv")
df_activity_log = pd.read_csv("test/activity_log_test.csv")
df_students = pd.read_csv("test/students_test.csv")
df_task_marks = pd.read_csv("test/task_marks_test.csv")
df_courses = pd.read_csv("test/courses_test.csv")
df_labels = pd.read_csv("test/final_marks_test.csv")

In [50]:
df_labels['final_mark'] = (np.round(df_labels['final_mark'] / 10)).astype(int) # Adjust final marks.

student_ids_in_labels = df_labels['student_id'].unique() # Filter students that are not marked at all. 
filtered_students = df_students[df_students['student_id'].isin(student_ids_in_labels)]
filtered_students = filtered_students.drop_duplicates(subset='student_id', keep='first') # There were duplicated, keep the first, drop the second

In [51]:
# Students Pre processing
def students_preprocess(df):
    # DOB to age. forgot to normalize it tho
    df["dob"] = pd.to_datetime(df["dob"])
    today = pd.Timestamp(datetime.today().date())
    df["age"] = df["dob"].apply(lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day)))
    df["age"] = scaler.fit_transform(df[["age"]])
    df.drop(columns=["dob"], inplace=True)

    # hot encoding of nationality and gender
    encoder = OneHotEncoder(sparse_output=False)  
    encoded = encoder.fit_transform(df[["gender", "nationality"]])

    encoded_cols = encoder.get_feature_names_out(["gender", "nationality"])
    df_encoded = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)

    # merging
    df_final = pd.concat([df.drop(columns=["gender", "nationality"]), df_encoded], axis=1)
    return df_final
df_pp_students = students_preprocess(filtered_students)

In [52]:
# NEW - Breadth and intensity of study/engagement a student has with non graded tasks.
def course_task_non_grade_revised(df_courses_tasks, df_activity_log):
    """
    Computes student-level features based on engagement with non-graded tasks,
    but aggregated per course to maintain context.
    - non_graded_interactions_per_course: intensity of study
    - unique_non_graded_tasks_viewed_per_course: breadth of study
    
    Returns a dataframe with features for each student-course pair.
    """

    non_graded_tasks = df_courses_tasks[ # all non graded tasks - extra workload
        (df_courses_tasks["is_resource"] == True) | (df_courses_tasks["weight"] == 0)
    ]
    
    logs_non_graded = df_activity_log[ # filtering logs to see the interactions with these tasks
        df_activity_log["task_id"].isin(non_graded_tasks["task_id"])
    ].copy() # cop removes the warning
    
    if logs_non_graded.empty:
        # if no interaction, then return empty df accordingly
        return pd.DataFrame(columns=[
            "student_id",
            "course_id",
            "non_graded_interactions_per_course",
            "unique_non_graded_tasks_viewed_per_course"
        ])
    
    # Calculate the total number of interactions per student-course.
    activity_counts_per_course = ( # keep it consistent with the labels df, student_id, course_id merging
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .count()
        .reset_index(name="non_graded_interactions_per_course")
    )
    
    task_coverage_per_course = ( # count the number of unique tasks vewed per student
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .nunique()
        .reset_index(name="unique_non_graded_tasks_viewed_per_course")
    )

    features_per_course = activity_counts_per_course.merge( # merging the two new features
        task_coverage_per_course, on=["student_id", "course_id"], how="outer"
    )

    features_per_course = features_per_course.fillna(0) # if its nan fill it with 0 since they didnt interact anyway

    # will normalize later on
    return features_per_course
student_course_non_graded_features = course_task_non_grade_revised(df_courses_tasks, df_activity_log)

In [53]:
def get_interaction_features(task_marks, activity_log, courses_tasks):
    # Performance based
    performance_features = task_marks.groupby(['student_id', 'course_id']).agg( # performance based features
        avg_mark=('mark', 'mean'), # avg of the marks
        std_mark=('mark', 'std'), # std of the marks, removing one or the other affected the results so we kept both of them
        tasks_submitted=('task_id', 'count') # specific task ids and submittions
    ).reset_index()
    performance_features['std_mark'] = performance_features['std_mark'].fillna(0) # filling the nans(no submissions), with std, obviously it is better than avg so

    # Engagement based
    # getting engagement here, it differs from view and submit since some of the tasks are not graded but still gives some engagement and workload to student
    action_counts = activity_log.groupby(['student_id', 'course_id', 'action']).size().unstack(fill_value=0)
    action_counts = action_counts.rename(columns={'view': 'view_count', 'submit': 'submit_count'})

    # to make it easier on the model and get a better understanding of the data we ratio it 
    action_counts['total_activity'] = action_counts['view_count'] + action_counts['submit_count']
    # np.divide for faster and safer division
    action_counts['view_submit_ratio'] = np.divide(action_counts['view_count'], action_counts['submit_count'])
    action_counts['view_submit_ratio'] = action_counts['view_submit_ratio'].replace([np.inf, -np.inf], 0).fillna(0)
    engagement_features = action_counts.reset_index()
    
    # Timeliness based
    # stding the time format just in case
    activity_log['timestamp'] = pd.to_datetime(activity_log['timestamp'])
    courses_tasks['deadline'] = pd.to_datetime(courses_tasks['deadline'])

    submissions = activity_log[activity_log['action'] == 'submit'].copy() # only the submissions which is graded

    submission_deadlines = pd.merge( # merging submission with task deadlines
        submissions,
        courses_tasks[['task_id', 'deadline']],
        on='task_id',
        how='left'
    )
    #
    # calculate lateness in hours. positive means late, negative means early.
    submission_deadlines['lateness_hours'] = (submission_deadlines['timestamp'] - submission_deadlines['deadline']).dt.total_seconds() / 3600
    submission_deadlines['is_late'] = submission_deadlines['lateness_hours'] > 0
    
    timeliness_features = submission_deadlines.groupby(['student_id', 'course_id']).agg( #aggregating timeliness for each student x course
        avg_lateness_hours=('lateness_hours', 'mean'),
        max_lateness_hours=('lateness_hours', 'max'),
        num_late_submissions=('is_late', 'sum'),
        num_early_submissions=('is_late', lambda x: (1 - x).sum()) # early submissions count by inverting the is_late
    ).reset_index()
    # If a student was never late, max_lateness might be negative. Let's cap it at 0.
    timeliness_features['max_lateness_hours'] = timeliness_features['max_lateness_hours'].clip(lower=0) # if a student is never late then it gets negative which is no bueno, so cap is 0
    


    # Merging everything we got
    df_merged = pd.merge(performance_features, engagement_features, on=['student_id', 'course_id'], how='left')
    df_merged = pd.merge(df_merged, timeliness_features, on=['student_id', 'course_id'], how='left')

    # Fill any remaining NaNs with 0 (e.g., if a student has no submissions for timeliness)
    df_merged = df_merged.fillna(0)

    return df_merged
interaction_df = get_interaction_features(df_task_marks, df_activity_log,  df_courses_tasks)

In [54]:
def get_course_features(courses, courses_tasks):
    
    # Task and resource counts 
    # split the tasks into resource or assignment from is_resource
    task_counts = courses_tasks[courses_tasks['is_resource'] == False].groupby('course_id').size().rename('task_count')
    resource_counts = courses_tasks[courses_tasks['is_resource'] == True].groupby('course_id').size().rename('resource_count')

    # how hard/intense the course is
    course_features = courses[['course_id', 'ects', 'duration_months']].copy()

    # there was no courses with 0 duration but still just in case of test and hidden data later on.
    duration = course_features['duration_months'].replace(0, 1)
    course_features['course_intensity'] = course_features['ects'] / duration # straight up ects/duration of the course, this is how it is calculated IRL as well so, seems like a good fit
    
    # merging everything    
    course_features = course_features.merge(task_counts, on='course_id', how='left') # all is left to lose nothing
    course_features = course_features.merge(resource_counts, on='course_id', how='left')

    course_features = course_features.fillna(0) # if by chance there are no tasks for a course, dont wanna keep NaN during training
    
    # make srue everything is integer
    course_features['task_count'] = course_features['task_count'].astype(int)
    course_features['resource_count'] = course_features['resource_count'].astype(int)

    # Normalization lter like before
    return course_features


course_features_df = get_course_features(df_courses, df_courses_tasks)

In [55]:
# combine all the features accordingly student_id x course_id like in the final_marks
# again all is left to lose nothing
merged = interaction_df.merge(
    df_labels, 
    on=['student_id', 'course_id'], 
    how='left'  # or 'left' if you want to keep all course records
)
df_test = merged.merge(
    df_pp_students,
    on='student_id',
    how='left'  # or 'left' to keep all records
)
df_test = df_test.merge(
    course_features_df,
    on='course_id',
    how='left'  # or 'left' to keep all records
)
df_test = df_test.merge(
    student_course_non_graded_features,
    on=['student_id', 'course_id'],
    how='left'
)
df_test

,student_id,course_id,avg_mark,std_mark,tasks_submitted,submit_count,view_count,total_activity,view_submit_ratio,avg_lateness_hours,...,nationality_Spain,nationality_Sweden,nationality_USA,ects,duration_months,course_intensity,task_count,resource_count,non_graded_interactions_per_course,unique_non_graded_tasks_viewed_per_course
0,STU000151,CRSB0032F,0.0,0.0,1,0.0,3.0,3.0,0.0,0.0,...,0.0,0.0,0.0,9,4,2.25,2,6,3.0,3.0
1,STU002818,CRSC68ECB,0.0,0.0,1,0.0,3.0,3.0,0.0,0.0,...,0.0,0.0,0.0,10,4,2.50,2,6,3.0,3.0
2,STU007F8A,CRSC68ECB,37.0,0.0,1,1.0,4.0,5.0,4.0,192.0,...,1.0,0.0,0.0,10,4,2.50,2,6,3.0,3.0
3,STU00A806,CRSC44D41,0.0,0.0,1,0.0,2.0,2.0,0.0,0.0,...,0.0,0.0,0.0,9,4,2.25,2,5,2.0,2.0
4,STU00DFE8,CRS1DEABD,78.0,0.0,1,1.0,1.0,2.0,1.0,-48.0,...,0.0,0.0,0.0,6,4,1.50,2,1,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,STUFE1F14,CRS80F602,44.0,0.0,1,1.0,3.0,4.0,3.0,48.0,...,0.0,0.0,0.0,3,4,0.75,2,2,2.0,1.0
716,STUFE24C1,CRS0AF493,62.0,0.0,1,1.0,2.0,3.0,2.0,96.0,...,0.0,0.0,1.0,9,4,2.25,2,2,1.0,1.0
717,STUFE24C1,CRS29E4BA,88.0,0.0,1,1.0,4.0,5.0,4.0,-120.0,...,0.0,0.0,1.0,5,1,5.00,4,4,3.0,3.0
718,STUFEF777,CRS29E4BA,77.0,0.0,1,1.0,4.0,5.0,4.0,48.0,...,0.0,0.0,1.0,5,1,5.00,4,4,3.0,3.0


In [56]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

X = df_test.drop(columns=["student_id", "course_id", "final_mark"])
y = df_test["final_mark"]
model = joblib.load('best_model.pkl')
predictions = model.predict(X)

# Calculate individual metrics
accuracy = accuracy_score(y, predictions)
precision = precision_score(y, predictions, average='weighted', zero_division=0)
recall = recall_score(y, predictions, average='weighted', zero_division=0)
f1 = f1_score(y, predictions, average='weighted', zero_division=0)

# Print the results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.5833
Precision: 0.5792
Recall: 0.5833
F1-Score: 0.5788
